# Skill Gap Engine
Comparing each employee's current skills against their role's required skills from O*NET to identify gaps.

In [1]:
import pandas as pd
import os

attr = pd.read_csv('../data/processed/employee_attrition_processed.csv')
emp_skills = pd.read_csv('../data/processed/employee_skills_current.csv')
role_master = pd.read_csv('../data/processed/role_master.csv')

# Load O*NET essential skills (competencies required by role/title)
ess = pd.read_csv('../data/raw/essential_skills.csv')
print(f'Essential skills dataset: {ess.shape}')
role_required_map = ess.groupby('Title')['Element Name'].apply(lambda x: list(x.unique())[:5]).to_dict()

print(f'Unique O*NET titles with skills: {len(role_required_map)}')

merged = attr[['EmployeeID', 'JobRole']].merge(emp_skills[['EmployeeID', 'current_skills']], on='EmployeeID', how='left')

def compute_gap(row):
    jobrole = str(row['JobRole'])
    current = set(str(row.get('current_skills', '')).split(', '))
    
    required = set(role_required_map.get(jobrole, []))
    if not required:
        for title in role_required_map:
            if any(word in title.lower() for word in jobrole.lower().split() if len(word) > 4):
                required = set(role_required_map[title][:5])
                break
    
    gap = required - current
    return ', '.join(list(gap)[:3]) if gap else 'No skill gap identified'

merged['MissingSkills'] = merged.apply(compute_gap, axis=1)
merged.to_csv('../data/processed/employee_skill_gaps.csv', index=False)

print(f'Skill gaps computed for {len(merged)} employees.')
print(merged[['EmployeeID', 'JobRole', 'MissingSkills']].head(10).to_string(index=False))

Essential skills dataset: (18200, 15)
Unique O*NET titles with skills: 910
Skill gaps computed for 500 employees.
 EmployeeID         JobRole                                    MissingSkills
          1         Auditor Active Listening, Reading Comprehension, Writing
          2 Sales Executive Active Listening, Reading Comprehension, Writing
          3        Helpdesk                          No skill gap identified
          4    HR Executive Active Listening, Reading Comprehension, Writing
          5 Account Manager Active Listening, Reading Comprehension, Writing
          6        Helpdesk                          No skill gap identified
          7        Engineer Active Listening, Reading Comprehension, Writing
          8    HR Executive Active Listening, Reading Comprehension, Writing
          9    HR Executive Active Listening, Reading Comprehension, Writing
         10      Accountant Active Listening, Reading Comprehension, Writing
